## LLMxCPG

In [ ]:
!git clone https://github.com/qcri/llmxcpg.git
%cd llmxcpg
!pip install -r requirements.txt
!pip install transformers
%cd ..

Cloning into 'llmxcpg'...
remote: Enumerating objects: 302, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 302 (delta 20), reused 14 (delta 6), pack-reused 262 (from 1)
Receiving objects: 100% (302/302), 44.06 MiB | 28.14 MiB/s, done.
Resolving deltas: 100% (84/84), done.
/content/llmxcpg
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 3.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.2/71.2 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 161.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━

/content


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL = "QCRI/LLMxCPG-Q"

# 1. Load Tokenizer
tok = AutoTokenizer.from_pretrained(MODEL)

# 2. Clean 8-bit Config
# No CPU offload needed, your GPU has plenty of room
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
)

# 3. Load Model
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb_config,
    device_map="auto",             # Will now comfortably assign everything to cuda:0
    torch_dtype=torch.bfloat16     # Essential for H100: keeps computation crazy fast
)

print(f"Model successfully loaded on: {model.device}")

config.json:   0%|          | 0.00/837 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.54k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/63.2k [00:00<?, ?B/s]

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

In [ ]:
# Get Java (Joern requires this)
!apt-get update -qq && apt-get install -y openjdk-11-jdk wget unzip

# Download the official install script
!wget https://github.com/joernio/joern/releases/latest/download/joern-install.sh -O joern-install.sh
!chmod +x joern-install.sh

# Run the install script non-interactively
# Default installs to ~/bin/joern
!bash ./joern-install.sh --yes

In [ ]:
import os

os.environ["PATH"] += os.pathsep + os.path.expanduser("~/bin/joern")
print(os.environ["PATH"])


In [ ]:
import os
import re  # <-- Imported the regex module

# Define the path to the directory containing your code files
DIRECTORY_PATH = "/content."

# Dictionary to store the extracted JSON (Key: filename, Value: JSON string or Error)
llm_results = {}

if not os.path.exists(DIRECTORY_PATH):
    print(f"Error: The directory '{DIRECTORY_PATH}' does not exist.")
else:
    print(f"Starting batch processing from: {DIRECTORY_PATH}\n")

    for filename in os.listdir(DIRECTORY_PATH):
        file_path = os.path.join(DIRECTORY_PATH, filename)

        if os.path.isfile(file_path):
            print(f"Processing '{filename}'...")

            with open(file_path, "r", encoding="utf-8") as f:
                code = f.read()
            run = """
Instruction:
Your task is to design Precise Joern CPGQL Queries for
Vulnerability Analysis.
Objective:
Develop targeted CPGQL Joern queries to:
• Identify taint flows based on your analysis.
• Capture potential vulnerability paths.
Constraints:
• Queries must be executable in Joern/CPGQL
• Use Scala language features for query construction
• Last query must use reachableByFlows to identify vulnerable paths
Output Requirements:
Do not provide any explanation apart from the desired CPGQL
Provide a JSON object with one field "queries": Sequence
of CPGQL queries to detect vulnerability
Expected JSON Output Format:
{
"queries": ["Query1" , "Query2", ..., "Final
Reachable Flows Query"]
}
Example Output:
{
  "queries": [
    "val freeCallsWithIdentifier = cpg.method.name(\"(.*_)?free\")
      .filter(_.parameter.size == 1)
      .callIn
      .where(_.argument(1).isIdentifier)
      ",
    "freeCallsWithIdentifier.flatMap(f => {
        val freedIdentifierCode = f.argument(1).code
        val postDom              = f.postDominatedBy.toSetImmutable
        val assignedPostDom = postDom.isIdentifier
          .where(_.inAssignment)
          .codeExact(freedIdentifierCode)
          .flatMap(id => id ++ id.postDominatedBy)
        postDom
          .removedAll(assignedPostDom)
          .isIdentifier
          .codeExact(freedIdentifierCode)
          .reachableByFlows(f.argument(1))
      })"
  ]
}

Input:

""" + code
            # --- 1. LLM EXECUTION LOGIC ---
            inputs = tok(run, return_tensors="pt", truncation=True, max_length=2048)
            inputs = {k: v.to(model.device) for k, v in inputs.items()}

            outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.0, do_sample=False)

            input_length = inputs['input_ids'].shape[1]
            generated_tokens = outputs[0][input_length:]

            # The raw output string from the LLM
            output_str = tok.decode(generated_tokens, skip_special_tokens=True)

            # --- 2. REGEX JSON EXTRACTION ---
            # Using non-greedy match to grab the first JSON block
            match = re.search(r'\{.*?\}', output_str, re.DOTALL)

            if match:
                clean_json = match.group(0)
                print(f"  ✅ Extracted JSON successfully.")
                llm_results[filename] = clean_json
            else:
                print(f"  ❌ Failed to find JSON.")
                # Store a fallback value so you know which files failed parsing
                llm_results[filename] = "ERROR_NO_JSON_FOUND"

    print("\n✅ Processing complete!")
    print(f"Successfully processed {len(llm_results)} files.")

# Preview the results for the first file
if llm_results:
    first_file = list(llm_results.keys())[0]
    print(f"\nPreview of '{first_file}' JSON extraction:\n{llm_results[first_file]}")

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Starting batch processing from: /content/fin

Processing 'CWE36_Absolute_Path_Traversal__wchar_t_environment_ifstream_33.cpp'...


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  ✅ Extracted JSON successfully.
Processing 'CWE36_Absolute_Path_Traversal__wchar_t_connect_socket_ifstream_16.cpp'...
  ✅ Extracted JSON successfully.

✅ Processing complete!
Successfully processed 2 files.

Preview of 'CWE36_Absolute_Path_Traversal__wchar_t_environment_ifstream_33.cpp' JSON extraction:
{
  "queries": ['val source = cpg.call.name("GETENV").argument.codeExact("ENV_VARIABLE").l', 'val sink = cpg.call.name("open").argument.l', 'sink.reachableByFlows(source)']
}


### regex updated

In [ ]:
import os
import re  # <-- Imported the regex module

# Define the path to the directory containing your code files
DIRECTORY_PATH = "/content."

# Dictionary to store the extracted JSON (Key: filename, Value: JSON string or Error)
llm_results = {}

if not os.path.exists(DIRECTORY_PATH):
    print(f"Error: The directory '{DIRECTORY_PATH}' does not exist.")
else:
    print(f"Starting batch processing from: {DIRECTORY_PATH}\n")

    for filename in os.listdir(DIRECTORY_PATH):
        file_path = os.path.join(DIRECTORY_PATH, filename)

        if os.path.isfile(file_path):
            print(f"Processing '{filename}'...")

            with open(file_path, "r", encoding="utf-8") as f:
                code = f.read()
            run = """
Instruction:
Your task is to design Precise Joern CPGQL Queries for
Vulnerability Analysis.
Objective:
Develop targeted CPGQL Joern queries to:
• Identify taint flows based on your analysis.
• Capture potential vulnerability paths.
Constraints:
• Queries must be executable in Joern/CPGQL
• Use Scala language features for query construction
• Last query must use reachableByFlows to identify vulnerable paths
Output Requirements:
Do not provide any explanation apart from the desired CPGQL
Provide a JSON object with one field "queries": Sequence
of CPGQL queries to detect vulnerability
Expected JSON Output Format:
{
"queries": ["Query1" , "Query2", ..., "Final
Reachable Flows Query"]
}
Example Output:
{
  "queries": [
    "val freeCallsWithIdentifier = cpg.method.name(\"(.*_)?free\")
      .filter(_.parameter.size == 1)
      .callIn
      .where(_.argument(1).isIdentifier)
      ",
    "freeCallsWithIdentifier.flatMap(f => {
        val freedIdentifierCode = f.argument(1).code
        val postDom              = f.postDominatedBy.toSetImmutable
        val assignedPostDom = postDom.isIdentifier
          .where(_.inAssignment)
          .codeExact(freedIdentifierCode)
          .flatMap(id => id ++ id.postDominatedBy)
        postDom
          .removedAll(assignedPostDom)
          .isIdentifier
          .codeExact(freedIdentifierCode)
          .reachableByFlows(f.argument(1))
      })"
  ]
}

Input:

""" + code
            # --- 1. LLM EXECUTION LOGIC ---
            inputs = tok(run, return_tensors="pt", truncation=True, max_length=2048)
            inputs = {k: v.to(model.device) for k, v in inputs.items()}

            outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.0, do_sample=False)

            input_length = inputs['input_ids'].shape[1]
            generated_tokens = outputs[0][input_length:]

            # The raw output string from the LLM
            output_str = tok.decode(generated_tokens, skip_special_tokens=True)

            # --- 2. REGEX JSON EXTRACTION ---
            # Using non-greedy match to grab the first JSON block
           # --- 2. REGEX JSON EXTRACTION ---
            # Extract anything between curly braces
            match = re.search(r'\{.*?\}', output_str, re.DOTALL)

            if match:
                clean_json = match.group(0)

                # NEW VALIDATION STEP: Make sure it actually looks like our expected JSON
                if '"queries"' in clean_json or "'queries'" in clean_json:
                    print(f"  ✅ Extracted JSON successfully.")
                    llm_results[filename] = clean_json
                else:
                    print(f"  ❌ LLM Hallucinated code instead of JSON.")
                    llm_results[filename] = "ERROR_NO_JSON_FOUND"
            else:
                print(f"  ❌ Failed to find JSON.")
                llm_results[filename] = "ERROR_NO_JSON_FOUND"

    print("\n✅ Processing complete!")
    print(f"Successfully processed {len(llm_results)} files.")

# Preview the results for the first file
if llm_results:
    first_file = list(llm_results.keys())[0]
    print(f"\nPreview of '{first_file}' JSON extraction:\n{llm_results[first_file]}")

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Starting batch processing from: /content/fin

Processing 'CWE36_Absolute_Path_Traversal__wchar_t_environment_ifstream_33.cpp'...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  ✅ Extracted JSON successfully.
Processing 'CWE36_Absolute_Path_Traversal__wchar_t_connect_socket_ifstream_16.cpp'...


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  ✅ Extracted JSON successfully.
Processing 'CWE78_OS_Command_Injection__wchar_t_console_system_02_queries.json'...


KeyboardInterrupt: 

## Joern


### *data error


In [ ]:
import os
import subprocess

# Paths setup
DIRECTORY_PATH = "/content."          # Where your original source codes live
OUTPUT_DIR = "/content."              # Where Joern results will be saved
JOERN_SCRIPT_PATH = "joernscript.py"   # Make sure this is uploaded to your Colab workspace

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Check if your joern script is uploaded
if not os.path.exists(JOERN_SCRIPT_PATH):
    print(f"⚠️ Error: '{JOERN_SCRIPT_PATH}' not found. Please upload it to your Colab environment.")
else:
    print(f"Starting Joern batch processing. Outputs will be saved to: {OUTPUT_DIR}\n")

    # Iterate through the dictionary from the previous cell
    for filename, json_content in llm_results.items():
        if json_content == "ERROR_NO_JSON_FOUND":
            print(f"⏭️ Skipping '{filename}' (No valid JSON was generated by the LLM).")
            continue

        print(f"\n--- Processing '{filename}' with Joern ---")

        # Define specific file paths for this iteration
        source_file = os.path.join(DIRECTORY_PATH, filename)

        # Remove the file extension for cleaner output filenames
        base_name = os.path.splitext(filename)[0]
        queries_file = os.path.join(OUTPUT_DIR, f"{base_name}_queries.json")
        cpg_file = os.path.join(OUTPUT_DIR, f"{base_name}_cpg.bin")
        flow_file = os.path.join(OUTPUT_DIR, f"{base_name}_flows.txt")
        snippet_file = os.path.join(OUTPUT_DIR, f"{base_name}_snippet.json")

        # ==========================================
        # OPTION 2: THE QUICK-FIX SANITIZATION
        # Intercepts dangling regex meta-characters
        # and converts them to exact string matches
        # ==========================================
        json_content = json_content.replace('.code("*', '.codeExact("*')
        json_content = json_content.replace('.code(\'*', '.codeExact(\'*') # Handles single quotes just in case

        # Write the LLM-generated JSON string to a file for joernscript.py to read
        with open(queries_file, "w", encoding="utf-8") as f:
            f.write(json_content)

        # Construct the execution command
        cmd = [
            "python", JOERN_SCRIPT_PATH,
            source_file,
            queries_file,
            cpg_file,
            flow_file,
            snippet_file
        ]

        # Execute the script via subprocess
        result = subprocess.run(cmd, capture_output=True, text=True)

        # Check execution status
        if result.returncode == 0:
            print(f"✅ Successfully processed '{filename}'.")
            print(f"   Final snippet JSON saved to: {snippet_file}")
            # Optional: Print standard output if you want to see the Joern logs
            # print(result.stdout)
        else:
            print(f"❌ Error processing '{filename}'.")
            print(f"Error details:\n{result.stderr}")

    print("\n🏁 All Joern processing completed!")

Starting Joern batch processing. Outputs will be saved to: /content/fin


--- Processing 'CWE121_Stack_Based_Buffer_Overflow__CWE129_listen_socket_33.cpp' with Joern ---
❌ Error processing 'CWE121_Stack_Based_Buffer_Overflow__CWE129_listen_socket_33.cpp'.
Error details:
Traceback (most recent call last):
  File "/content/joernscript.py", line 513, in <module>
    main()
  File "/content/joernscript.py", line 480, in main
    run_cmd(f'joern --script "{script_path}"')
  File "/content/joernscript.py", line 21, in run_cmd
    raise RuntimeError(result.stderr)
RuntimeError: executing /tmp/tmps5v40nd0.sc
-- Error: /tmp/wrapped-script1382870951657244866.sc:5:56 -----------------------
  5 |def getSinks = cpg.method.name("bad").call.code("buffer\[data\] = 1")
    |                                                        ^
    |                                              invalid escape character
-- Error: /tmp/wrapped-script1382870951657244866.sc:5:62 -----------------------
  5 |def getSink

### regular joern run

In [ ]:
import os
import subprocess

# Paths setup
DIRECTORY_PATH = "/content."          # Where your original source codes live
OUTPUT_DIR = "/content."          # Where Joern results will be saved
JOERN_SCRIPT_PATH = "joernscript.py"   # Make sure this is uploaded to your Colab workspace

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Check if your joern script is uploaded
if not os.path.exists(JOERN_SCRIPT_PATH):
    print(f"⚠️ Error: '{JOERN_SCRIPT_PATH}' not found. Please upload it to your Colab environment.")
else:
    print(f"Starting Joern batch processing. Outputs will be saved to: {OUTPUT_DIR}\n")

    # Iterate through the dictionary from the previous cell
    for filename, json_content in llm_results.items():
        if json_content == "ERROR_NO_JSON_FOUND":
            print(f"⏭️ Skipping '{filename}' (No valid JSON was generated by the LLM).")
            continue

        print(f"\n--- Processing '{filename}' with Joern ---")

        # Define specific file paths for this iteration
        source_file = os.path.join(DIRECTORY_PATH, filename)

        # Remove the file extension for cleaner output filenames
        base_name = os.path.splitext(filename)[0]
        queries_file = os.path.join(OUTPUT_DIR, f"{base_name}_queries.json")
        cpg_file = os.path.join(OUTPUT_DIR, f"{base_name}_cpg.bin")
        flow_file = os.path.join(OUTPUT_DIR, f"{base_name}_flows.txt")
        snippet_file = os.path.join(OUTPUT_DIR, f"{base_name}_snippet.json")

        # Write the LLM-generated JSON string to a file for joernscript.py to read
        with open(queries_file, "w", encoding="utf-8") as f:
            f.write(json_content)

        # Construct the execution command
        cmd = [
            "python", JOERN_SCRIPT_PATH,
            source_file,
            queries_file,
            cpg_file,
            flow_file,
            snippet_file
        ]

        # Execute the script via subprocess
        result = subprocess.run(cmd, capture_output=True, text=True)

        # Check execution status
        if result.returncode == 0:
            print(f"✅ Successfully processed '{filename}'.")
            print(f"   Final snippet JSON saved to: {snippet_file}")
            # Optional: Print standard output if you want to see the Joern logs
            # print(result.stdout)
        else:
            print(f"❌ Error processing '{filename}'.")
            print(f"Error details:\n{result.stderr}")

    print("\n🏁 All Joern processing completed!")

Starting Joern batch processing. Outputs will be saved to: /content/fin


--- Processing 'CWE36_Absolute_Path_Traversal__wchar_t_environment_ifstream_33.cpp' with Joern ---
✅ Successfully processed 'CWE36_Absolute_Path_Traversal__wchar_t_environment_ifstream_33.cpp'.
   Final snippet JSON saved to: /content/fin/CWE36_Absolute_Path_Traversal__wchar_t_environment_ifstream_33_snippet.json

--- Processing 'CWE36_Absolute_Path_Traversal__wchar_t_connect_socket_ifstream_16.cpp' with Joern ---
✅ Successfully processed 'CWE36_Absolute_Path_Traversal__wchar_t_connect_socket_ifstream_16.cpp'.
   Final snippet JSON saved to: /content/fin/CWE36_Absolute_Path_Traversal__wchar_t_connect_socket_ifstream_16_snippet.json

🏁 All Joern processing completed!


## Store JSON files

In [ ]:
import os
import shutil

# Paths setup
SOURCE_DIR = "/content."     # Where your files currently are
DEST_DIR = "/content."       # The new folder for just the JSON files

# Ensure the destination directory exists
os.makedirs(DEST_DIR, exist_ok=True)

print(f"Scanning '{SOURCE_DIR}' for snippet JSON files...\n")

copied_count = 0

# Check if the source directory exists
if not os.path.exists(SOURCE_DIR):
    print(f"⚠️ Error: Source directory '{SOURCE_DIR}' does not exist.")
else:
    # Iterate through all files in the source directory
    for filename in os.listdir(SOURCE_DIR):

        # KEY CHANGE: Only match files that end specifically with _snippet.json
        if filename.endswith("_snippet.json"):
            source_file_path = os.path.join(SOURCE_DIR, filename)
            dest_file_path = os.path.join(DEST_DIR, filename)

            # Copy the file to the new destination
            shutil.copy(source_file_path, dest_file_path)

            print(f"📄 Copied: {filename}")
            copied_count += 1

    print(f"\n✅ Done! Successfully copied {copied_count} snippet files to '{DEST_DIR}'.")

Scanning '/content/fin' for snippet JSON files...

📄 Copied: CWE36_Absolute_Path_Traversal__wchar_t_connect_socket_ifstream_16_snippet.json
📄 Copied: CWE36_Absolute_Path_Traversal__wchar_t_environment_ifstream_33_snippet.json

✅ Done! Successfully copied 2 snippet files to '/content/fo'.


In [ ]:
import os
import json

# The folder containing your final snippet JSONs
# Change this if you want to target a different directory
DIRECTORY_PATH = "/content."

print(f"Scanning '{DIRECTORY_PATH}' to clean JSON files...\n")

cleaned_count = 0

# Check if the directory exists
if not os.path.exists(DIRECTORY_PATH):
    print(f"⚠️ Error: Directory '{DIRECTORY_PATH}' does not exist.")
else:
    # Iterate through all files in the directory
    for filename in os.listdir(DIRECTORY_PATH):

        # Only process .json files
        if filename.endswith(".json"):
            file_path = os.path.join(DIRECTORY_PATH, filename)

            try:
                # 1. Read the JSON data
                with open(file_path, "r", encoding="utf-8") as f:
                    data = json.load(f)

                # 2. Remove the keys
                # Using pop(key, None) safely removes the key without crashing
                # if the key happens to already be missing
                data.pop("source_file", None)
                data.pop("cpg_path", None)

                # 3. Write the cleaned data back to the file
                with open(file_path, "w", encoding="utf-8") as f:
                    json.dump(data, f, indent=2, ensure_ascii=False)

                print(f"🧹 Cleaned: {filename}")
                cleaned_count += 1

            except json.JSONDecodeError:
                print(f"❌ Error: '{filename}' contains invalid JSON and could not be read.")
            except Exception as e:
                print(f"❌ Error processing '{filename}': {e}")

    print(f"\n✅ Done! Successfully removed paths from {cleaned_count} JSON files.")

Scanning '/content/fo' to clean JSON files...

🧹 Cleaned: CWE36_Absolute_Path_Traversal__wchar_t_connect_socket_ifstream_16_snippet.json
🧹 Cleaned: CWE36_Absolute_Path_Traversal__wchar_t_environment_ifstream_33_snippet.json

✅ Done! Successfully removed paths from 2 JSON files.


In [ ]:
import shutil
import os

def zip_folder(folder_path, output_filename):
    """
    Compresses a directory into a zip archive.

    :param folder_path: Path to the folder you want to zip (e.g., './my_folder')
    :param output_filename: The name/path of the output file without the .zip extension (e.g., './backup')
    """
    if not os.path.exists(folder_path):
        print(f"Error: The folder '{folder_path}' does not exist.")
        return

    try:
        # The 'zip' argument specifies the format
        shutil.make_archive(output_filename, 'zip', folder_path)
        print(f"✅ Successfully zipped '{folder_path}' into '{output_filename}.zip'")
    except Exception as e:
        print(f"❌ An error occurred: {e}")

# --- Example Usage ---
# Replace these strings with your actual folder path and desired zip name
TARGET_FOLDER = "/content."
OUTPUT_ZIP_NAME = "./bite"

zip_folder(TARGET_FOLDER, OUTPUT_ZIP_NAME)

✅ Successfully zipped '/content/fo' into './bite.zip'
